# Claude Code Generation Example

This notebook demonstrates how to use the `ClaudeCodeAssistant` to generate trading strategy code using Anthropic's Claude API or template-based generation.

## Setup

First, ensure you have the required packages installed and an API key configured.

In [ ]:
# Install required packages
!pip install anthropic pandas numpy

In [ ]:
# Set your Anthropic API key
import os
os.environ['ANTHROPIC_API_KEY'] = 'your-api-key-here'

# Or pass it directly
API_KEY = os.environ.get('ANTHROPIC_API_KEY', 'sk-ant-...')

## Import Required Modules

In [ ]:
from cbsc_strategy_sdk.claude import (
    ClaudeCodeAssistant,
    StrategyType,
    GenerationMode,
)
import pandas as pd
import numpy as np

## Initialize Assistant

Create a `ClaudeCodeAssistant` instance. Set `fallback_to_template=True` to use template-based generation when the API is unavailable.

In [ ]:
# Initialize assistant
assistant = ClaudeCodeAssistant(
    api_key=API_KEY,
    model="claude-3-5-sonnet-20241022",
    max_tokens=4096,
    fallback_to_template=True,  # Use templates if API unavailable
)

# Check API availability
print(f"API Available: {assistant.is_api_available()}")

## Generate Strategy Using Template

First, let's generate a strategy using the template system (no API call required).

In [ ]:
# Generate momentum strategy using template
result = assistant.generate_strategy(
    description="Simple momentum strategy with 20-day lookback",
    strategy_type=StrategyType.MOMENTUM,
    mode=GenerationMode.TEMPLATE,
)

print(f"Generated by: {result.mode_used.value}")
print(f"Valid: {result.is_valid}")
print(f"\nCode:\n{'-'*60}")
print(result.code)

## Generate Strategy with Custom Parameters

In [ ]:
# Generate with custom parameters
custom_params = {
    "name": "CustomMomentum",
    "lookback_period": 15,
    "threshold": 0.03,
    "position_size": 0.2,
    "use_rsi": True,
    "rsi_period": 14,
}

result = assistant.generate_strategy(
    description="Momentum with RSI filter",
    strategy_type=StrategyType.MOMENTUM,
    parameters=custom_params,
    mode=GenerationMode.TEMPLATE,
)

print(result.code)

## Generate Different Strategy Types

In [ ]:
# Mean Reversion Strategy
mean_rev_result = assistant.generate_strategy(
    description="Mean reversion with Bollinger Bands",
    strategy_type=StrategyType.MEAN_REVERSION,
    parameters={
        "use_bollinger": True,
        "bb_period": 20,
        "bb_std": 2.0,
    },
    mode=GenerationMode.TEMPLATE,
)

print("Mean Reversion Strategy:")
print(mean_rev_result.code[:500])  # Show first 500 chars

In [ ]:
# Pair Trading Strategy
pair_result = assistant.generate_strategy(
    description="Cointegrated pair trading",
    strategy_type=StrategyType.PAIR_TRADING,
    parameters={
        "pair": ("XOM", "CVX"),
        "lookback_window": 30,
    },
    mode=GenerationMode.TEMPLATE,
)

print("Pair Trading Strategy:")
print(pair_result.code[:500])

In [ ]:
# ML Strategy
ml_result = assistant.generate_strategy(
    description="Random Forest classifier strategy",
    strategy_type=StrategyType.ML_STRATEGY,
    parameters={
        "model_type": "rf",
        "features": ["returns", "volume", "volatility"],
    },
    mode=GenerationMode.TEMPLATE,
)

print("ML Strategy:")
print(ml_result.code[:500])

## Test Generated Strategy

Let's test one of the generated strategies with sample data.

In [ ]:
# Generate sample price data
np.random.seed(42)
dates = pd.date_range('2024-01-01', periods=252, freq='D')
prices = 100 + np.cumsum(np.random.randn(252) * 0.5)

data = pd.DataFrame({
    'close': prices,
    'open': prices + np.random.randn(252) * 0.1,
    'high': prices + np.abs(np.random.randn(252) * 0.2),
    'low': prices - np.abs(np.random.randn(252) * 0.2),
    'volume': np.random.randint(1000000, 5000000, 252),
}, index=dates)

print(f"Sample data shape: {data.shape}")
data.head()

In [ ]:
# Execute the generated strategy code
exec_globals = {}
exec(result.code, exec_globals)

# Get the strategy class
strategy_class_name = [k for k in exec_globals.keys() if k[0].isupper()][0]
StrategyClass = exec_globals[strategy_class_name]

# Instantiate strategy
strategy = StrategyClass()

# Generate signals
signals = strategy.generate_signals(data)
print(f"\nSignal distribution:")
print(signals.value_counts())

In [ ]:
# Run backtest
backtest_results = strategy.backtest(data)

print(f"Final Equity: ${backtest_results['final_equity']:,.2f}")
print(f"Total Return: {backtest_results['total_return']:.2%}")
print(f"Sharpe Ratio: {backtest_results['sharpe_ratio']:.2f}")
print(f"Max Drawdown: {backtest_results['max_drawdown']:.2%}")

## Generate with Claude API (if available)

If you have a valid API key, you can use Claude's AI to generate more sophisticated strategies.

In [ ]:
# Try API-based generation (will fall back to template if API unavailable)
api_result = assistant.generate_strategy(
    description="""Create a momentum strategy that:
    1. Uses multiple timeframe analysis (daily and weekly)
    2. Incorporates volatility-adjusted position sizing
    3. Has built-in stop-loss at 2%
    4. Only takes long positions
    """,
    strategy_type=StrategyType.MOMENTUM,
    mode=GenerationMode.HYBRID,  # Try API, fall back to template
)

print(f"Generated by: {api_result.mode_used.value}")
print(f"\n{api_result.code[:800]}...")  # Show first 800 chars

## Parameter Optimization

Get suggestions for optimizing strategy parameters based on backtest results.

In [ ]:
# Analyze performance and get optimization suggestions
performance_metrics = {
    "sharpe_ratio": backtest_results['sharpe_ratio'],
    "total_return": backtest_results['total_return'],
    "max_drawdown": backtest_results['max_drawdown'],
    "win_rate": 0.52,
}

# Note: This requires API access for intelligent suggestions
# With template-only mode, will return basic structure
suggestions = assistant.optimize_parameters(
    code=result.code,
    performance_metrics=performance_metrics,
)

print("Optimization Suggestions:")
print(f"Suggested Parameters: {suggestions.suggested_parameters}")
print(f"Rationale: {suggestions.rationale}")
print(f"Confidence: {suggestions.confidence:.0%}")

## Error Analysis

Get help fixing errors in strategy code.

In [ ]:
# Example: Get help with a syntax error
broken_code = '''
class BrokenStrategy:
    def __init__(self):
        self.period = 20
    
    def generate_signals(self, data):
        # Missing return statement
        momentum = data['close'].pct_change(self.period)
        # This will cause an error
'''

# Analyze the error
fix = assistant.analyze_errors(
    code=broken_code,
    error_message="NameError: name 'pd' is not defined",
)

print("Root Cause:", fix.root_cause)
print("\nPrevention Tips:")
for tip in fix.prevention_tips:
    print(f"  - {tip}")

## Usage Statistics

Track API usage and costs.

In [ ]:
# Get usage statistics
usage = assistant.get_usage_stats()

if usage:
    print(f"Total Requests: {usage.total_requests}")
    print(f"Input Tokens: {usage.input_tokens:,}")
    print(f"Output Tokens: {usage.output_tokens:,}")
    print(f"Total Tokens: {usage.total_tokens:,}")
    print(f"Estimated Cost: ${usage.total_cost_usd:.4f}")
else:
    print("No API usage statistics available (template mode)")

## Summary

This notebook demonstrated:

1. **Template-based generation**: Generate strategies without API calls
2. **Custom parameters**: Customize generated strategies with parameters
3. **Multiple strategy types**: Momentum, mean reversion, pair trading, ML
4. **Testing**: Execute and backtest generated strategies
5. **Optimization**: Get parameter improvement suggestions (with API)
6. **Error analysis**: Get help fixing code issues

### Next Steps

- Experiment with different strategy types and parameters
- Use the API for more sophisticated, customized strategies
- Integrate generated strategies into your trading workflow
- Combine multiple generated strategies for ensemble approaches